In [1]:
#Test data
from pathlib import Path

DATA_DIR = Path("student_data")
DATA_DIR.mkdir(exist_ok=True)

sample_files = {
    "class_a.csv": (
        "student_id,name,math,english,science,computer\n"
        "101,Ali Khan,85,78,90,88\n"
        "102,Sara Ahmed,92,88,95,91\n"
        "103,Bilal Raza,40,35,50,45\n"
    ),
    "class_b.csv": (
        "student_id,name,math,english,science,computer\n"
        "201,Hina Malik,70,65,80,75\n"
        "202,Usman Tariq,abc,70,60,55\n"      # marks text hain
        "203,,80,80,80,80\n"                  # name missing
        "204,Zara Iqbal,150,90,85,80\n"       # marks 100 se zyada
        "205,Omar Farooq,60,,70,65\n"         # english missing
        "101,Ali Khan,85,78,90,88\n"          # duplicate ID
    ),
    "class_d.csv": "student_id,name,math\n301,Nida,80\n",   # columns kam
    "empty.csv": "",
}

for fname, content in sample_files.items():
    (DATA_DIR / fname).write_text(content, encoding="utf-8")

# Corrupted (binary garbage) file
(DATA_DIR / "corrupted.csv").write_bytes(b"\x00\xff\xfe\x80\x81garbage\x00\x00")

print("Sample files ban gayi:", sorted(p.name for p in DATA_DIR.iterdir()))

Sample files ban gayi: ['class_a.csv', 'class_b.csv', 'class_d.csv', 'corrupted.csv', 'empty.csv']


In [2]:
#Config + safe helper functions
import csv
import io

REQUIRED_COLS = ["student_id", "name", "math", "english", "science", "computer"]
SUBJECTS = REQUIRED_COLS[2:]
PASS_PERCENTAGE = 40

# In files ka hona zaroori hai (missing hui to flag hongi)
EXPECTED_FILES = ["class_a.csv", "class_b.csv", "class_c.csv",
                  "class_d.csv", "corrupted.csv", "empty.csv"]


def read_csv_safely(path):
    """File parh kar rows (list of dict) do, warna (None, error)."""
    try:
        if not path.exists():
            return None, "file missing"
        if path.stat().st_size == 0:
            return None, "file empty"

        text = path.read_text(encoding="utf-8-sig")
        if "\x00" in text:
            return None, "corrupted (binary data)"

        reader = csv.DictReader(io.StringIO(text))
        if not reader.fieldnames:
            return None, "no header row"

        headers = [h.strip().lower() for h in reader.fieldnames if h]
        missing = [c for c in REQUIRED_COLS if c not in headers]
        if missing:
            return None, f"missing columns: {', '.join(missing)}"

        rows = []
        for raw in reader:
            row = {}
            for k, v in raw.items():
                if k is None:          # extra columns ignore
                    continue
                row[k.strip().lower()] = (v or "").strip()
            rows.append(row)
        return rows, None

    except UnicodeDecodeError:
        return None, "corrupted (unreadable encoding)"
    except PermissionError:
        return None, "permission denied"
    except csv.Error as e:
        return None, f"csv format error: {e}"
    except Exception as e:                       # koi bhi aur masla
        return None, f"unexpected error: {type(e).__name__}: {e}"


def validate_student(row):
    """Row check karo. (clean_dict, None) ya (None, reason)."""
    sid = row.get("student_id", "")
    name = row.get("name", "")
    if not sid:
        return None, "missing student_id"
    if not name:
        return None, "missing name"

    marks = {}
    for subject in SUBJECTS:
        raw = row.get(subject, "")
        if raw == "":
            return None, f"{subject}: marks missing"
        try:
            value = float(raw)
        except ValueError:
            return None, f"{subject}: not a number ('{raw}')"
        if not (0 <= value <= 100):              # NaN/inf bhi yahin pakray jaate hain
            return None, f"{subject}: out of range ({raw})"
        marks[subject] = value

    total = sum(marks.values())
    percentage = round(total / (100 * len(SUBJECTS)) * 100, 2)
    return {
        "student_id": sid,
        "name": name.title(),
        **marks,
        "total": total,
        "percentage": percentage,
        "grade": get_grade(percentage),
        "status": "PASS" if percentage >= PASS_PERCENTAGE else "FAIL",
    }, None


def get_grade(p):
    if p >= 80: return "A"
    if p >= 70: return "B"
    if p >= 60: return "C"
    if p >= 50: return "D"
    return "F"

In [3]:
#Saari files process karein
clean_results = []
flagged = []        # failed files + failed students dono
file_log = []       # har file ka status
seen_ids = set()


def process_all():
    # Folder mein maujood CSVs + expected files ka union
    found = {p.name for p in DATA_DIR.glob("*.csv")} if DATA_DIR.exists() else set()
    all_files = sorted(found | set(EXPECTED_FILES))

    for fname in all_files:
        rows, error = read_csv_safely(DATA_DIR / fname)

        if error:
            flagged.append({"type": "FILE", "source_file": fname, "row": "-",
                            "student_id": "-", "name": "-", "reason": error})
            file_log.append((fname, "FAILED", error))
            continue

        ok = bad = 0
        for i, row in enumerate(rows, start=2):      # row 1 = header
            try:
                record, reason = validate_student(row)
                if record and record["student_id"] in seen_ids:
                    record, reason = None, "duplicate student_id"

                if record:
                    seen_ids.add(record["student_id"])
                    record["source_file"] = fname
                    clean_results.append(record)
                    ok += 1
                else:
                    flagged.append({"type": "STUDENT", "source_file": fname, "row": i,
                                    "student_id": row.get("student_id", "") or "?",
                                    "name": row.get("name", "") or "?",
                                    "reason": reason})
                    bad += 1
            except Exception as e:                   # row level safety net
                flagged.append({"type": "STUDENT", "source_file": fname, "row": i,
                                "student_id": "?", "name": "?",
                                "reason": f"unexpected error: {e}"})
                bad += 1

        file_log.append((fname, "OK", f"{ok} valid, {bad} flagged"))


try:
    process_all()
except Exception as e:                               # aakhri safety net
    print(f"⚠️ Processing mein masla aaya lekin program chal raha hai: {e}")

print(f"Valid students: {len(clean_results)} | Flagged items: {len(flagged)}")

Valid students: 4 | Flagged items: 9


In [4]:
#Output files
def write_csv_safely(path, rows, fieldnames):
    try:
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        return True
    except Exception as e:
        print(f"❌ {path} likhi nahi ja saki: {e}")
        return False


result_cols = ["student_id", "name", *SUBJECTS, "total", "percentage",
               "grade", "status", "source_file"]
report_cols = ["type", "source_file", "row", "student_id", "name", "reason"]

sorted_results = sorted(clean_results, key=lambda r: r["percentage"], reverse=True)

write_csv_safely("clean_results.csv", sorted_results, result_cols)
write_csv_safely("failed_report.csv", flagged, report_cols)
print("✅ clean_results.csv aur failed_report.csv ban gayi")

✅ clean_results.csv aur failed_report.csv ban gayi


In [5]:
#Neat summary
print("=" * 62)
print("STUDENT RESULT PROCESSOR - SUMMARY".center(62))
print("=" * 62)

print("\nFILE STATUS")
print("-" * 62)
for fname, status, info in file_log:
    icon = "✅" if status == "OK" else "❌"
    print(f"  {icon} {fname:<16} {info}")

print("\nTOP RESULTS")
print("-" * 62)
for r in sorted_results:
    print(f"  {r['student_id']:<6} {r['name']:<14} {r['percentage']:>6.2f}%  "
          f"Grade {r['grade']}  {r['status']}")

print("\nFLAGGED")
print("-" * 62)
for f in flagged:
    who = f["name"] if f["type"] == "STUDENT" else "(whole file)"
    print(f"  [{f['type']}] {f['source_file']} row {f['row']}: {who} -> {f['reason']}")

passed = sum(1 for r in clean_results if r["status"] == "PASS")
print("\n" + "=" * 62)
print(f"Valid: {len(clean_results)} | Passed: {passed} | "
      f"Failed: {len(clean_results) - passed} | Flagged: {len(flagged)}")
print("=" * 62)

              STUDENT RESULT PROCESSOR - SUMMARY              

FILE STATUS
--------------------------------------------------------------
  ✅ class_a.csv      3 valid, 0 flagged
  ✅ class_b.csv      1 valid, 5 flagged
  ❌ class_c.csv      file missing
  ❌ class_d.csv      missing columns: english, science, computer
  ❌ corrupted.csv    corrupted (unreadable encoding)
  ❌ empty.csv        file empty

TOP RESULTS
--------------------------------------------------------------
  102    Sara Ahmed      91.50%  Grade A  PASS
  101    Ali Khan        85.25%  Grade A  PASS
  201    Hina Malik      72.50%  Grade B  PASS
  103    Bilal Raza      42.50%  Grade F  PASS

FLAGGED
--------------------------------------------------------------
  [STUDENT] class_b.csv row 3: Usman Tariq -> math: not a number ('abc')
  [STUDENT] class_b.csv row 4: ? -> missing name
  [STUDENT] class_b.csv row 5: Zara Iqbal -> math: out of range (150)
  [STUDENT] class_b.csv row 6: Omar Farooq -> english: marks missing
